# BNR-PE Dual-Axis Research Walkthrough (Colab)

This notebook is a pedagogical demo of BNR-PE behavior in single-axis vs dual-axis regimes.

## Learning goals
1. Understand how BNR-PE applies structured low-rank rotations.
2. Verify key invariants (norm preservation) in practice.
3. Measure runtime overhead across axis profiles and ranks.
4. Build intuition for where the method is efficient vs expensive.


## 0) Environment setup

If you are in Colab, this cell clones the repo and installs requirements.


In [ ]:
import os
import subprocess
from pathlib import Path

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    if not Path('/content/BNRPE').exists():
        subprocess.run(['git', 'clone', 'https://github.com/fyremael/BNRPE.git', '/content/BNRPE'], check=True)
    subprocess.run(['python', '-m', 'pip', 'install', '-r', '/content/BNRPE/bnrpe_jax/requirements.txt'], check=True)
    os.chdir('/content/BNRPE/bnrpe_jax')
else:
    # Assumes notebook is run from repo context locally.
    if Path('bnrpe').exists():
        pass
    elif Path('../bnrpe').exists():
        os.chdir('..')

print('Working directory:', Path.cwd())


## 1) Imports and helpers


In [ ]:
import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

from bnrpe.params import init_params
from bnrpe.rotors import apply_bnrpe_batch

def apply_rope(X, pos):
    d = X.shape[-1]
    inv_freq = 1.0 / (10000 ** (jnp.arange(0, d, 2, dtype=jnp.float32) / d))
    theta = pos[:, None] * inv_freq[None, :]
    c, s = jnp.cos(theta), jnp.sin(theta)
    x_even, x_odd = X[:, 0::2], X[:, 1::2]
    y_even = x_even * c - x_odd * s
    y_odd = x_even * s + x_odd * c
    Y = jnp.empty_like(X)
    Y = Y.at[:, 0::2].set(y_even)
    Y = Y.at[:, 1::2].set(y_odd)
    return Y

def build_positions(length, profile='single_axis'):
    t = jnp.arange(length, dtype=jnp.float32)
    if profile == 'single_axis':
        return t, jnp.stack([t, jnp.zeros_like(t)], axis=-1)
    if profile == 'dual_axis_non_degenerate':
        axis1 = 0.5 * t + 1.0 + 0.25 * jnp.sin(t * 0.03125)
        return t, jnp.stack([t, axis1], axis=-1)
    raise ValueError(profile)

def timed_steady(fn, *args, iters=20):
    f = jax.jit(fn)
    out = f(*args)
    out.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(iters):
        out = f(*args)
    out.block_until_ready()
    return (time.perf_counter() - t0) / iters


## 2) Invariant check: norm preservation

BNR-PE is constructed from Cayley transforms and should preserve vector norms up to numerical tolerance.


In [ ]:
L, d, r = 256, 128, 8
key = jax.random.PRNGKey(0)
params = init_params(jax.random.PRNGKey(1), d=d, r=r, n_axes=2, alpha=0.2)
X = jax.random.normal(key, (L, d))

profiles = ['single_axis', 'dual_axis_non_degenerate']
rows = []
for p in profiles:
    _, P = build_positions(L, p)
    Y = apply_bnrpe_batch(X, P, params)
    n0 = jnp.linalg.norm(X, axis=-1)
    n1 = jnp.linalg.norm(Y, axis=-1)
    err = jnp.abs(n1 - n0)
    rows.append({'profile': p, 'mean_abs_norm_err': float(jnp.mean(err)), 'max_abs_norm_err': float(jnp.max(err))})

pd.DataFrame(rows)


## 3) Mini benchmark: RoPE vs BNR-PE

We benchmark overhead for rank 4 and rank 8 in both position profiles.


In [ ]:
L, d = 256, 128
X = jax.random.normal(jax.random.PRNGKey(3), (L, d))
results = []

for profile in ['single_axis', 'dual_axis_non_degenerate']:
    pos, P = build_positions(L, profile)
    rope_t = timed_steady(apply_rope, X, pos, iters=20)
    for rank in [4, 8]:
        params = init_params(jax.random.PRNGKey(100 + rank), d=d, r=rank, n_axes=2, alpha=0.2)
        bnr_t = timed_steady(lambda X_, P_: apply_bnrpe_batch(X_, P_, params), X, P, iters=20)
        overhead = 100.0 * (bnr_t / rope_t - 1.0)
        results.append({
            'profile': profile,
            'rank': rank,
            'rope_s': rope_t,
            'bnr_s': bnr_t,
            'overhead_pct': overhead,
        })

df = pd.DataFrame(results)
df


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
pivot = df.pivot(index='rank', columns='profile', values='overhead_pct')
pivot.plot(kind='bar', ax=ax, color=['#94a3b8', '#0ea5e9'])
ax.set_title('BNR-PE Overhead vs RoPE (L=256, d=128)')
ax.set_ylabel('Overhead %')
ax.axhline(0.0, color='black', linewidth=1)
ax.legend(title='Profile')
plt.tight_layout()
plt.show()


## 4) Matrix sweep for intuition

This creates a small rank-8 dual-axis surface over `(length, dim)`.


In [ ]:
lengths = [128, 256, 512]
dims = [64, 128, 256, 512]
matrix = []

for L in lengths:
    X = jax.random.normal(jax.random.PRNGKey(1000 + L), (L, max(dims)))
    pos, P = build_positions(L, 'dual_axis_non_degenerate')
    for d in dims:
        Xd = X[:, :d]
        rope_t = timed_steady(apply_rope, Xd, pos, iters=15)
        params = init_params(jax.random.PRNGKey(2000 + L + d), d=d, r=8, n_axes=2, alpha=0.2)
        bnr_t = timed_steady(lambda X_, P_: apply_bnrpe_batch(X_, P_, params), Xd, P, iters=15)
        matrix.append({'length': L, 'dim': d, 'overhead_pct': 100.0 * (bnr_t / rope_t - 1.0)})

m = pd.DataFrame(matrix)
m


In [ ]:
heat = m.pivot(index='length', columns='dim', values='overhead_pct')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(heat.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(list(heat.columns))
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(list(heat.index))
ax.set_xlabel('dim')
ax.set_ylabel('length')
ax.set_title('Rank-8 Dual-Axis Overhead Surface')
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        ax.text(j, i, f"{heat.values[i, j]:.1f}", ha='center', va='center', color='white', fontsize=9)
fig.colorbar(im, ax=ax, label='overhead %')
plt.tight_layout()
plt.show()


## 5) Interpretation prompts

1. Which `(length, dim)` regimes remain expensive for rank-8 dual-axis?
2. Where does overhead become moderate or low?
3. If you optimize next, do you target low-width kernels or change rank/profile assumptions?

## Suggested extensions
- Repeat sweep for multiple seeds and plot confidence bands.
- Compare exact path vs hybrid path from `scripts/prototype_fused_paths.py`.
- Reproduce CI thresholds and gate logic with `scripts/build_gate_report.py`.
